# 01. Fock States, Ladder Operators & Hilbert Space Truncation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_1_states_of_light/01_fock_states_and_ladder_operators.ipynb)

* **Module:** 1 — Single-Mode Quantum States of Light
* **Difficulty:** Intermediate
* **Estimated Time:** 45 minutes
* **Prerequisites:** Module 0 (especially Notebooks 0.2 and 0.4 on Dirac notation and Harmonic Oscillator).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Represent discrete single-mode Fock states** $|n\rangle$ and ladder operators ($\hat{a}, \hat{a}^\dagger, \hat{n}$) in truncated Hilbert spaces using QuTiP.
2. **Analyze the mathematical effects of Hilbert space truncation** ($N < \infty$): prove why the canonical commutator $[\hat{a}, \hat{a}^\dagger] = \mathbb{I}$ develops a boundary artifact on the last diagonal element $-(N-1)$.
3. **Calculate photon statistics of Fock states**: verify zero photon number variance $\langle (\Delta n)^2 \rangle = 0$ (definite photon number).
4. **Formulate continuous optical field quadratures** $\hat{X}_1 = \frac{1}{2}(\hat{a} + \hat{a}^\dagger)$ and $\hat{X}_2 = \frac{1}{2i}(\hat{a} - \hat{a}^\dagger)$ and evaluate the phase-insensitive quadrature noise $\langle (\Delta X_1)^2 \rangle = \langle (\Delta X_2)^2 \rangle = \frac{2n+1}{4}$.
5. **Establish numerical convergence criteria**: determine how large the truncation dimension $N$ must be to accurately simulate quantum states with mean photon count $\bar{n}$.


---
## 0. Setup & Dependencies

We will use **QuTiP (Quantum Toolbox in Python)** alongside NumPy and Matplotlib.


In [ ]:
# Install dependencies in Google Colab (uncomment if running in Colab)
# !pip install -q qutip numpy scipy matplotlib ipywidgets

import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import math

# QuTiP import
try:
    import qutip as qt
    from qutip import basis, destroy, create, num, coherent, qeye
    has_qutip = True
    print(f"QuTiP version {qt.__version__} initialized.")
except ImportError:
    has_qutip = False
    print("QuTiP not installed locally. Using internal NumPy fallback.")

# Configure clean, publication-ready plotting aesthetics
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})


---
## 1. The Fock Basis & Ladder Operators

In quantum optics, the fundamental building blocks of single-mode radiation fields are the **Fock states** (or **number states**), denoted by $|n\rangle$, where $n \in \{0, 1, 2, \dots\}$ represents an exact, integer number of photons in the cavity mode.

The Fock states form a complete, orthonormal basis for the single-mode Hilbert space $\mathcal{H}$:

$$\langle m | n \rangle = \delta_{mn}$$

$$\sum_{n=0}^{\infty} |n\rangle\langle n| = \mathbb{I}$$

### The Ladder Operators
The mode is governed by the bosonic annihilation operator $\hat{a}$ and creation operator $\hat{a}^\dagger$:

$$\hat{a}|n\rangle = \sqrt{n}|n-1\rangle \quad (\hat{a}|0\rangle = 0)$$

$$\hat{a}^\dagger|n\rangle = \sqrt{n+1}|n+1\rangle$$

The **photon number operator** is the Hermitian product:

$$\hat{n} \equiv \hat{a}^\dagger \hat{a}, \quad \hat{n}|n\rangle = n|n\rangle$$

### Matrix Representation in Truncated Dimension $N$
Because computer memory is finite, an infinite-dimensional Hilbert space must be truncated to a cutoff dimension $N$. The basis is truncated to $\{|0\rangle, |1\rangle, \dots, |N-1\rangle\}$.

The ladder operators become $N \times N$ matrices:

$$\hat{a} = \begin{pmatrix} 0 & \sqrt{1} & 0 & \dots & 0 \\ 0 & 0 & \sqrt{2} & \dots & 0 \\ 0 & 0 & 0 & \ddots & \vdots \\ \vdots & \vdots & \vdots & \ddots & \sqrt{N-1} \\ 0 & 0 & 0 & \dots & 0 \end{pmatrix}, \quad \hat{a}^\dagger = \begin{pmatrix} 0 & 0 & \dots & 0 \\ \sqrt{1} & 0 & \dots & 0 \\ 0 & \sqrt{2} & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \sqrt{N-1} & 0 \end{pmatrix}$$

Notice that $\hat{a}$ is an upper-diagonal matrix with entries $\sqrt{n}$ on the superdiagonal, while $\hat{a}^\dagger$ has entries $\sqrt{n}$ on the subdiagonal.


In [ ]:
def get_ladder_operators(N: int):
    """
    Returns the truncated N x N annihilation, creation, and number operators.
    """
    if has_qutip:
        return destroy(N), create(N), num(N)
    else:
        # NumPy fallback
        a = np.zeros((N, N), dtype=complex)
        for k in range(1, N):
            a[k-1, k] = np.sqrt(k)
        a_dag = a.conj().T
        n = a_dag @ a
        return a, a_dag, n

# Inspect matrix representations for N = 6
N_demo = 6
a_demo, adag_demo, n_demo = get_ladder_operators(N_demo)

a_mat = a_demo.full() if has_qutip else a_demo
adag_mat = adag_demo.full() if has_qutip else adag_demo
n_mat = n_demo.full() if has_qutip else n_demo

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4.5))

im1 = ax1.imshow(np.abs(a_mat), cmap="Blues", vmin=0, vmax=np.sqrt(N_demo))
ax1.set_title(r"Annihilation Operator $\hat{a}$")
ax1.set_xlabel("Column Index")
ax1.set_ylabel("Row Index")
for (i, j), val in np.ndenumerate(a_mat):
    if abs(val) > 1e-4:
        ax1.text(j, i, f"{val.real:.2f}", ha="center", va="center", color="white", fontweight="bold")

im2 = ax2.imshow(np.abs(adag_mat), cmap="Blues", vmin=0, vmax=np.sqrt(N_demo))
ax2.set_title(r"Creation Operator $\hat{a}^\dagger$")
ax2.set_xlabel("Column Index")
ax2.set_ylabel("Row Index")
for (i, j), val in np.ndenumerate(adag_mat):
    if abs(val) > 1e-4:
        ax2.text(j, i, f"{val.real:.2f}", ha="center", va="center", color="white", fontweight="bold")

im3 = ax3.imshow(np.abs(n_mat), cmap="Blues", vmin=0, vmax=N_demo - 1)
ax3.set_title(r"Number Operator $\hat{n} = \hat{a}^\dagger\hat{a}$")
ax3.set_xlabel("Column Index")
ax3.set_ylabel("Row Index")
for (i, j), val in np.ndenumerate(n_mat):
    if abs(val) > 1e-4:
        ax3.text(j, i, f"{int(val.real)}", ha="center", va="center", color="white", fontweight="bold")

plt.tight_layout()
plt.show()


---
## 2. The Bosonic Commutator & Truncation Boundary Artifacts

In an infinite-dimensional Hilbert space, the bosonic ladder operators satisfy the exact canonical commutation relation:

$$[\hat{a}, \hat{a}^\dagger] = \hat{a}\hat{a}^\dagger - \hat{a}^\dagger\hat{a} = \mathbb{I}$$

However, in any finite $N$-dimensional truncated basis, a fascinating anomaly occurs. Recall from linear algebra that the trace of any matrix commutator is strictly zero:

$$\mathrm{Tr}([\hat{a}, \hat{a}^\dagger]) = \mathrm{Tr}(\hat{a}\hat{a}^\dagger) - \mathrm{Tr}(\hat{a}^\dagger\hat{a}) \equiv 0$$

How can $[\hat{a}, \hat{a}^\dagger] \approx \mathbb{I}$ when its trace must be zero?

### Calculating Matrix Elements of the Commutator
Evaluating the diagonal products:
* $(\hat{a}^\dagger \hat{a})_{nn} = n \quad (0 \le n \le N-1)$
* $(\hat{a} \hat{a}^\dagger)_{nn} = n + 1 \quad (0 \le n \le N-2)$
* At the truncated boundary $n = N-1$: the creation operator cannot generate state $|N\rangle$, so $(\hat{a} \hat{a}^\dagger)_{N-1, N-1} = 0$.

Subtracting the two matrices:

$$[\hat{a}, \hat{a}^\dagger] = \begin{pmatrix} 1 & 0 & \dots & 0 & 0 \\ 0 & 1 & \dots & 0 & 0 \\ \vdots & \vdots & \ddots & \vdots & \vdots \\ 0 & 0 & \dots & 1 & 0 \\ 0 & 0 & \dots & 0 & -(N-1) \end{pmatrix}$$

Notice that:
$$\mathrm{Tr}([\hat{a}, \hat{a}^\dagger]) = \underbrace{1 + 1 + \dots + 1}_{(N-1)\text{ times}} - (N-1) = 0$$

The first $N-1$ diagonal elements are **strictly $1.0$**! The entire trace deficit is concentrated entirely in the single boundary element $-(N-1)$ at the truncation edge.


In [ ]:
# Compute the truncated commutator for N = 10
N_comm = 10
a_c, adag_c, _ = get_ladder_operators(N_comm)

if has_qutip:
    comm_mat = (a_c * adag_c - adag_c * a_c).full()
else:
    comm_mat = a_c @ adag_c - adag_c @ a_c

diag_comm = np.diag(comm_mat).real

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# 1. Bar plot of diagonal elements
indices = np.arange(N_comm)
colors = ["#1f77b4"] * (N_comm - 1) + ["#d62728"]
ax1.bar(indices, diag_comm, color=colors, width=0.55)
ax1.axhline(1.0, color="green", ls="--", lw=1.5, label="Expected Commutator = 1.0")
ax1.axhline(0.0, color="gray", lw=1)
ax1.set_title(rf"Diagonal Elements of $[\hat{{a}}, \hat{{a}}^\dagger]$ ($N={N_comm}$)")
ax1.set_xlabel("Fock State Index $n$")
ax1.set_ylabel(r"$[\hat{a}, \hat{a}^\dagger]_{nn}$")
ax1.set_xticks(indices)
ax1.legend()

# 2. 2D Heatmap
im2 = ax2.imshow(comm_mat.real, cmap="coolwarm", vmin=-(N_comm - 1), vmax=1.5)
ax2.set_title(r"2D Heatmap of Matrix $[\hat{a}, \hat{a}^\dagger]$")
ax2.set_xlabel("Column Index")
ax2.set_ylabel("Row Index")
fig.colorbar(im2, ax=ax2, label="Value")

plt.tight_layout()
plt.show()

print(f"Sum of diagonal elements Tr([a, a_dag]): {np.sum(diag_comm):.4e} (Identically 0!)")
print(f"Boundary element [N-1, N-1]: {diag_comm[-1]:.1f} (Expected: -{N_comm-1})")


> **Physical Insight:** This provides the fundamental guideline for all quantum optical simulations: **quantum states must never populate Fock states near the truncation boundary $N-1$**. As long as state populations $\rho_{nn}$ are negligible near the boundary, the unphysical $-(N-1)$ artifact will never be sampled, and the simulation acts as an exact infinite-dimensional Hilbert space.


---
## 3. Optical Field Quadratures & Phase-Insensitive Noise

In classical optics, a monochromatic light wave is characterized by its electric field amplitude $E(t) = E_1 \cos(\omega t) + E_2 \sin(\omega t)$.

In quantum optics, the dimensionless counterparts to the classical in-phase ($E_1$) and out-of-phase ($E_2$) components are the **Hermitian Field Quadrature Operators**:

$$\hat{X}_1 = \frac{\hat{a} + \hat{a}^\dagger}{2}, \quad \hat{X}_2 = \frac{\hat{a} - \hat{a}^\dagger}{2i}$$

Using $[\hat{a}, \hat{a}^\dagger] = 1$, the commutation relation between quadratures is:

$$[\hat{X}_1, \hat{X}_2] = \frac{1}{4i}[\hat{a}+\hat{a}^\dagger, \hat{a}-\hat{a}^\dagger] = \frac{1}{4i}(-[\hat{a}, \hat{a}^\dagger] + [\hat{a}^\dagger, \hat{a}]) = \frac{i}{2}$$

By the Robertson-Schrödinger uncertainty relation:

$$\Delta X_1 \Delta X_2 \ge \frac{1}{2}\left| \langle [\hat{X}_1, \hat{X}_2] \rangle \right| = \frac{1}{4}$$

### Quadrature Statistics of Fock States $|n\rangle$
Let us compute the expectation values and variances for an arbitrary Fock state $|n\rangle$:

1. **Mean values:**
   $$\langle n | \hat{X}_1 | n \rangle = \frac{1}{2}\left( \langle n | \hat{a} | n \rangle + \langle n | \hat{a}^\dagger | n \rangle \right) = 0$$
   $$\langle n | \hat{X}_2 | n \rangle = 0$$
   Fock states possess **zero average electric and magnetic field**!

2. **Variances:**
   $$\hat{X}_1^2 = \frac{1}{4}(\hat{a}^2 + \hat{a}^{\dagger 2} + \hat{a}\hat{a}^\dagger + \hat{a}^\dagger\hat{a})$$
   Since $\langle n | \hat{a}^2 | n \rangle = 0$, and $\hat{a}\hat{a}^\dagger = \hat{n} + 1$:
   $$\langle n | \hat{X}_1^2 | n \rangle = \frac{1}{4}\langle n | (2\hat{n} + 1) | n \rangle = \frac{2n + 1}{4}$$

Therefore, both quadrature variances are identical:

$$\langle (\Delta X_1)^2 \rangle = \langle (\Delta X_2)^2 \rangle = \frac{2n + 1}{4}$$

* For the **Vacuum State** ($n=0$): $\Delta X_1 = \Delta X_2 = \frac{1}{2} \implies \Delta X_1 \Delta X_2 = \frac{1}{4}$. The vacuum is a **minimum uncertainty state**!
* For **Higher Fock States** ($n \ge 1$): $\Delta X_1 \Delta X_2 = \frac{2n+1}{4} > \frac{1}{4}$. Noise is symmetric across all phases (**phase-insensitive noise**), but grows linearly with photon number $n$.


In [ ]:
def compute_quadrature_uncertainties(N_max: int = 6):
    """
    Computes quadrature variances <(Delta X1)^2> and <(Delta X2)^2> for Fock states |0> through |N_max-1>.
    """
    N_dim = N_max + 10
    a_op, adag_op, _ = get_ladder_operators(N_dim)
    
    # Quadrature operators
    X1 = 0.5 * (a_op + adag_op)
    X2 = -0.5j * (a_op - adag_op)
    
    n_list = list(range(N_max))
    var_X1_list = []
    var_X2_list = []
    
    for n in n_list:
        if has_qutip:
            psi_n = basis(N_dim, n)
            mean_x1 = qt.expect(X1, psi_n)
            mean_x1_sq = qt.expect(X1 * X1, psi_n)
            var_x1 = mean_x1_sq - mean_x1**2
            
            mean_x2 = qt.expect(X2, psi_n)
            mean_x2_sq = qt.expect(X2 * X2, psi_n)
            var_x2 = mean_x2_sq - mean_x2**2
        else:
            psi_vec = np.zeros(N_dim, dtype=complex)
            psi_vec[n] = 1.0
            mean_x1 = (psi_vec.conj() @ (X1 @ psi_vec)).real
            mean_x1_sq = (psi_vec.conj() @ (X1 @ (X1 @ psi_vec))).real
            var_x1 = mean_x1_sq - mean_x1**2
            
            mean_x2 = (psi_vec.conj() @ (X2 @ psi_vec)).real
            mean_x2_sq = (psi_vec.conj() @ (X2 @ (X2 @ psi_vec))).real
            var_x2 = mean_x2_sq - mean_x2**2
            
        var_X1_list.append(var_x1)
        var_X2_list.append(var_x2)
        
    return n_list, var_X1_list, var_X2_list

n_indices, var1, var2 = compute_quadrature_uncertainties(N_max=6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 1. Variance vs n
ax1.plot(n_indices, var1, 'bo-', lw=2, markersize=8, label=r"Numerical $\langle (\Delta X_1)^2 \rangle$")
ax1.plot(n_indices, [(2*n + 1)/4.0 for n in n_indices], 'r--', lw=1.8, label=r"Analytical $\frac{2n+1}{4}$")
ax1.axhline(0.25, color="gray", ls=":", label="Vacuum Shot-Noise Limit (0.25)")
ax1.set_title("Quadrature Variance vs. Photon Number $n$")
ax1.set_xlabel("Photon Number $n$")
ax1.set_ylabel("Variance")
ax1.set_xticks(n_indices)
ax1.legend(loc="upper left")

# 2. Phase-space noise uncertainty circles in (X1, X2) plane
theta = np.linspace(0, 2*np.pi, 200)
colors_circle = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for n in range(4):
    radius = np.sqrt(var1[n])  # standard deviation radius
    ax2.plot(radius * np.cos(theta), radius * np.sin(theta), color=colors_circle[n], lw=2.0,
             label=rf"$|{n}\rangle$ ($\Delta X = {radius:.2f}$)")
    ax2.fill(radius * np.cos(theta), radius * np.sin(theta), color=colors_circle[n], alpha=0.08)

ax2.set_aspect("equal")
ax2.axhline(0, color="gray", lw=1)
ax2.axvline(0, color="gray", lw=1)
ax2.set_title("Phase Space Quadrature Noise Circles in $(X_1, X_2)$")
ax2.set_xlabel(r"Quadrature $X_1$")
ax2.set_ylabel(r"Quadrature $X_2$")
ax2.set_xlim(-1.8, 1.8)
ax2.set_ylim(-1.8, 1.8)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()


> **Physical Insight:** In the phase plane on the right, every Fock state $|n\rangle$ is a centered circle. This signifies that Fock states have a completely randomized optical phase $\phi \in [0, 2\pi]$: we know the photon number with absolute certainty ($\Delta n = 0$), so by the number-phase uncertainty principle ($\Delta n \Delta \phi \ge 1/2$), the phase is completely undetermined!


---
## 4. Hilbert Space Truncation Convergence Benchmark

In realistic computational quantum optics, we rarely simulate pure low-$n$ Fock states; instead, we simulate coherent states, squeezed states, or thermal states which are infinite superpositions over all Fock states:

$$|\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^{\infty} \frac{\alpha^n}{\sqrt{n!}} |n\rangle$$

The mean photon number is $\bar{n} = |\alpha|^2$, and the standard deviation is $\Delta n = \sqrt{\bar{n}}$.

### The Practical Rule of Thumb
Because a coherent state obeys Poissonian statistics, states beyond $\bar{n} + 4\sigma_n$ have negligible probability. A safe cutoff dimension $N_{\text{cutoff}}$ must satisfy:

$$N_{\text{cutoff}} \ge \bar{n} + 4\sqrt{\bar{n}} + 10$$

If $N$ is chosen too small:
1. The state norm is truncated: $\langle \psi | \psi \rangle < 1$.
2. State fidelity $F = |\langle \psi_{\text{true}} | \psi_N \rangle|^2$ plummets.
3. Evolution operators reflect unphysically off the $N-1$ truncation boundary.

Let us test this convergence systematically for coherent states with $\alpha = 1.5, 2.5, 3.5$ ($\bar{n} = 2.25, 6.25, 12.25$).


In [ ]:
def test_cutoff_convergence():
    alpha_values = [1.5, 2.5, 3.5]
    N_range = np.arange(5, 45, 2)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    
    for idx, alpha in enumerate(alpha_values):
        n_bar = alpha**2
        norm_errors = []
        fidelity_losses = []
        
        # High-dimension reference state (N = 80)
        N_ref = 80
        if has_qutip:
            psi_ref = coherent(N_ref, alpha)
        else:
            coeffs_ref = np.array([alpha**n / math.sqrt(float(math.factorial(n))) for n in range(N_ref)], dtype=complex)
            psi_ref = np.exp(-0.5 * abs(alpha)**2) * coeffs_ref
            
        for N_cut in N_range:
            if has_qutip:
                psi_cut = coherent(N_cut, alpha)
                # Norm error: 1 - <psi|psi>
                norm_sq = (psi_cut.dag() * psi_cut).tr().real
                norm_err = max(1e-16, abs(1.0 - norm_sq))
                
                # Fidelity with truncated slice of reference state
                overlap = abs((psi_ref.extract_states(list(range(N_cut))).dag() * psi_cut).tr())**2
                fid_loss = max(1e-16, abs(1.0 - overlap))
            else:
                coeffs = np.array([alpha**n / math.sqrt(float(math.factorial(n))) for n in range(N_cut)], dtype=complex)
                psi_cut = np.exp(-0.5 * abs(alpha)**2) * coeffs
                norm_sq = np.sum(np.abs(psi_cut)**2)
                norm_err = max(1e-16, abs(1.0 - norm_sq))
                fid_loss = norm_err
                
            norm_errors.append(norm_err)
            fidelity_losses.append(fid_loss)
            
        # Safe threshold rule: N_safe = n_bar + 4*sqrt(n_bar) + 10
        N_safe = int(n_bar + 4 * np.sqrt(n_bar) + 10)
        
        ax1.semilogy(N_range, norm_errors, 'o-', color=colors[idx], lw=1.8,
                     label=rf"$\alpha = {alpha}$ ($\bar{{n}} = {n_bar:.1f}$, $N_{{\mathrm{{safe}}}} \approx {N_safe}$)")
        ax1.axvline(N_safe, color=colors[idx], ls=":", alpha=0.7)
        
        # Plot photon distribution for context
        fock_k = np.arange(30)
        poiss_pk = np.exp(-n_bar) * (n_bar**fock_k) / np.array([math.factorial(k) for k in fock_k], dtype=float)
        ax2.plot(fock_k, poiss_pk, 'o-', color=colors[idx], lw=1.8, label=rf"$\bar{{n}} = {n_bar:.1f}$")
        
        ax1.set_title("Truncation Norm Error vs. Cutoff Dimension N")
    ax1.set_xlabel("Cutoff Dimension $N$")
    ax1.set_ylabel("Norm Error (Log Scale)")
    ax1.set_ylim(1e-16, 1.5)
    ax1.legend()
    
    ax2.set_title("Photon Number Distribution $P(n)$")
    ax2.set_xlabel("Photon Number $n$")
    ax2.set_ylabel("Probability $P(n)$")
    ax2.set_xlim(0, 30)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

test_cutoff_convergence()


> **Physical Insight:** Notice that as soon as the cutoff dimension $N$ exceeds the dotted vertical lines ($N_{\text{safe}} = \bar{n} + 4\sqrt{\bar{n}} + 10$), the truncation norm error drops precipitously below $10^{-14}$ (machine precision). When simulating any quantum optical protocol, always run a cutoff convergence test to ensure $N$ is chosen in the flat plateau regime!


---
## 5. Interactive Playground: Fock State Superposition Explorer

Unlike pure Fock states $|n\rangle$ which have zero expectation value for electric field ($\langle \hat{X}_1 \rangle = 0$), a **coherent superposition of adjacent Fock states**:

$$|\psi\rangle = \cos(\theta)|0\rangle + e^{i\phi}\sin(\theta)|1\rangle$$

exhibits a **non-zero macroscopic electric dipole field** $\langle \hat{X}_1 \rangle = \frac{1}{2}\sin(2\theta)\cos(\phi)$!

Use the interactive sliders below to explore how the mixing angle $\theta$ and relative phase $\phi$ generate electric field expectation values and shape the quadrature phase noise ellipse.


In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    has_widgets = True
except ImportError:
    has_widgets = False

def interactive_fock_superposition(theta_factor=0.25, phi_factor=0.0):
    theta = theta_factor * np.pi
    phi = phi_factor * np.pi
    
    N_dim = 15
    a_op, adag_op, _ = get_ladder_operators(N_dim)
    X1 = 0.5 * (a_op + adag_op)
    X2 = -0.5j * (a_op - adag_op)
    
    # State: cos(theta)|0> + exp(i*phi)*sin(theta)|1>
    c0 = np.cos(theta)
    c1 = np.exp(1j * phi) * np.sin(theta)
    
    if has_qutip:
        psi = c0 * basis(N_dim, 0) + c1 * basis(N_dim, 1)
        mean_x1 = qt.expect(X1, psi)
        mean_x2 = qt.expect(X2, psi)
        var_x1 = qt.expect(X1*X1, psi) - mean_x1**2
        var_x2 = qt.expect(X2*X2, psi) - mean_x2**2
    else:
        psi_v = np.zeros(N_dim, dtype=complex)
        psi_v[0] = c0
        psi_v[1] = c1
        mean_x1 = (psi_v.conj() @ (X1 @ psi_v)).real
        mean_x2 = (psi_v.conj() @ (X2 @ psi_v)).real
        var_x1 = (psi_v.conj() @ (X1 @ (X1 @ psi_v))).real - mean_x1**2
        var_x2 = (psi_v.conj() @ (X2 @ (X2 @ psi_v))).real - mean_x2**2
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    
    # 1. State Populations |c0|^2 and |c1|^2
    ax1.bar(["$|0\\rangle$", "$|1\\rangle$"], [abs(c0)**2, abs(c1)**2], color=["#1f77b4", "#ff7f0e"], width=0.45)
    ax1.set_ylim(0, 1.1)
    ax1.set_ylabel("Occupation Probability")
    ax1.set_title(rf"Fock Populations ($\theta = {theta_factor:.2f}\pi, \phi = {phi_factor:.2f}\pi$)")
    ax1.text(0, abs(c0)**2 + 0.04, f"{abs(c0)**2:.3f}", ha="center", fontweight="bold")
    ax1.text(1, abs(c1)**2 + 0.04, f"{abs(c1)**2:.3f}", ha="center", fontweight="bold")
    
    # 2. Phase-space displaced uncertainty circle
    circ_theta = np.linspace(0, 2*np.pi, 200)
    rx = np.sqrt(var_x1)
    ry = np.sqrt(var_x2)
    ax2.plot(mean_x1 + rx * np.cos(circ_theta), mean_x2 + ry * np.sin(circ_theta), color="#d62728", lw=2)
    ax2.fill(mean_x1 + rx * np.cos(circ_theta), mean_x2 + ry * np.sin(circ_theta), color="#d62728", alpha=0.15)
    ax2.scatter([mean_x1], [mean_x2], color="#d62728", s=60, label=rf"Mean $(\langle X_1 \rangle, \langle X_2 \rangle) = ({mean_x1:.2f}, {mean_x2:.2f})$")
    
    ax2.axhline(0, color="gray", lw=1)
    ax2.axvline(0, color="gray", lw=1)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.2, 1.2)
    ax2.set_ylim(-1.2, 1.2)
    ax2.set_title(r"Phase-Space Displacement in $(X_1, X_2)$")
    ax2.set_xlabel(r"Quadrature $\langle X_1 \rangle$")
    ax2.set_ylabel(r"Quadrature $\langle X_2 \rangle$")
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

if has_widgets:
    interact(
        interactive_fock_superposition,
        theta_factor=FloatSlider(min=0.0, max=0.5, step=0.05, value=0.25, description="Angle \u03b8/\u03c0:"),
        phi_factor=FloatSlider(min=0.0, max=2.0, step=0.1, value=0.0, description="Phase \u03c6/\u03c0:")
    )
else:
    print("Interactive widgets available in Jupyter/Colab. Displaying theta = pi/4, phi = 0:")
    interactive_fock_superposition(theta_factor=0.25, phi_factor=0.0)


---
## 6. Hands-On Exercises & Guided Challenges

### Exercise 1: Number State Variance vs. Poissonian Light
1. For any pure Fock state $|n\rangle$, prove analytically that the photon number variance:
   $$\langle (\Delta n)^2 \rangle = \langle \hat{n}^2 \rangle - \langle \hat{n} \rangle^2 = 0$$
2. Contrast this with classical Poissonian light (a coherent laser state), where variance equals the mean: $\langle (\Delta n)^2 \rangle = \bar{n}$. Explain why a Fock state $|n\rangle$ is an inherently non-classical state of light with sub-Poissonian statistics.

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Analytical proof:**
* $\hat{n}|n\rangle = n|n\rangle \implies \langle n | \hat{n} | n \rangle = n$.
* $\hat{n}^2|n\rangle = n^2|n\rangle \implies \langle n | \hat{n}^2 | n \rangle = n^2$.
* $\langle (\Delta n)^2 \rangle = n^2 - (n)^2 = 0$.

Because the photon number variance is strictly zero, photon counting measurements will *always* register exactly $n$ clicks without a single fluctuation. This represents the ultimate sub-Poissonian light source (Fano factor $F = 0$, Mandel $Q = -1$).
</details>

---

### Exercise 2: Truncation Dimension for a Squeezed Vacuum State
A squeezed vacuum state $|\xi\rangle = \hat{S}(r)|0\rangle$ with squeeze parameter $r = 1.0$ contains only even Fock states ($|0\rangle, |2\rangle, |4\rangle, \dots$).
Its mean photon number is $\bar{n} = \sinh^2(r) \approx \sinh^2(1.0) \approx 1.38$, but its photon distribution has a very long tail.
1. Write a short script in QuTiP to compute the state fidelity between a squeezed state constructed with cutoff $N$ and a reference state with $N=50$.
2. Determine the minimum $N$ required to achieve fidelity $F > 0.9999$.

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Code snippet:**
```python
r_squeeze = 1.0
if has_qutip:
    from qutip import squeeze
    psi_ref = squeeze(50, r_squeeze) * basis(50, 0)
    for N_test in range(5, 30, 2):
        psi_test = squeeze(N_test, r_squeeze) * basis(N_test, 0)
        overlap = abs((psi_ref.extract_states(list(range(N_test))).dag() * psi_test).tr())**2
        if overlap > 0.9999:
            print(f"Minimum cutoff N for 99.99% fidelity: N = {N_test}")
            break
```
</details>


---
## 7. Summary & What's Next

### Key Takeaways
1. **The Fock Basis:** Number states $|n\rangle$ are the exact energy eigenstates of the quantized optical mode. Ladder operators $\hat{a}$ and $\hat{a}^\dagger$ shift the photon count by $\pm 1$.
2. **Truncation Anomaly:** Truncating Hilbert space to dimension $N$ forces the trace of $[\hat{a}, \hat{a}^\dagger]$ to zero, creating an unphysical $-(N-1)$ element at the boundary. Keep state populations negligible near $N-1$ to prevent artifacts.
3. **Quadrature Uncertainties:** Fock states have zero mean electric field ($\langle \hat{X}_1 \rangle = \langle \hat{X}_2 \rangle = 0$) and phase-insensitive noise $\langle (\Delta X_1)^2 \rangle = \frac{2n+1}{4}$. The vacuum $|0\rangle$ is a minimum uncertainty state ($1/4$).
4. **Safe Cutoff Rule:** For states with mean photon number $\bar{n}$, always set $N_{\text{cutoff}} \ge \bar{n} + 4\sqrt{\bar{n}} + 10$ to ensure machine-precision convergence.

---

### What's Next?
In **Notebook 1.2 (`02_coherent_states_and_displacement.ipynb`)**, we will:
* Implement the Glauber **displacement operator** $\hat{D}(\alpha) = \exp(\alpha \hat{a}^\dagger - \alpha^* \hat{a})$.
* Generate **Coherent States** $|\alpha\rangle$ and prove they are right-eigenstates of the annihilation operator ($\hat{a}|\alpha\rangle = \alpha|\alpha\rangle$).
* Verify Poissonian photon counting statistics and minimum quadrature uncertainty $\Delta X_1 \Delta X_2 = 1/4$.

---
### References & Recommended Reading
* **Christopher Gerry & Peter Knight**, *Introductory Quantum Optics*, Chapter 2: Field Quantization and Number States.
* **Marlan O. Scully & M. Suhail Zubairy**, *Quantum Optics*, Chapter 1: Classical and Quantum Description of Light.
* **QuTiP Documentation**, *Basic Operations on Quantum Objects & Truncation Guidelines*.
